# Prompting language models with Transformers

## Learning goals

By the end of this notebook, you will be able to:

- Run a chat completion with Transformers' `text-generation` pipeline.
- Use a simple prompt to classify the sentiment of a social media post.
- Inspect the generated response and the pipeline's generation defaults.
- Limit generation to one token and enable greedy decoding.
- Retrieve token scores and logits, convert them to probabilities, and inspect alternatives.
- Connect next-token prediction to the BERT fill-mask exercise from Day 1.


<br><a target="_blank" href="https://colab.research.google.com/github/haukelicht/advanced_text_analysis/blob/main/notebooks/incontext_learning/llm_inference_basics_transformers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

::: {.callout-note title="Inference"}

In the context of working with LLMs, _inference_ means generating predictions or responses from a trained language model based on the input you provide.
We use an existing model without updating its weights.

:::


### The route taken by a prompt

Our input follows this route:

**Messages → chat template and tokenizer → model → decoding → assistant response**

The **pipeline** brings these steps together.
The **tokenizer** converts text into token IDs, and the **chat template** formats messages in the way the model expects.
The **model** computes scores for the next token; **decoding** determines which token to select.

There is no inference provider or API request in this workflow.
Loading files from the Hub can require a download, but generation itself runs locally.
See the [Transformers chat guide](https://huggingface.co/docs/transformers/conversations).

## Setup

### Python environment and model files

Use the course environment with `torch`, `transformers`, and `matplotlib` installed.
This notebook uses the `advanced_text_analysis` Jupyter kernel.
Select your installed course kernel if its name differs.


In [ ]:
COLAB = True
try:
    import google.colab
except ImportError:
    COLAB = False

if COLAB:
    !git clone --branch main --single-branch --depth 1 --filter=blob:none https://github.com/haukelicht/advanced_text_analysis.git
    %cd advanced_text_analysis

In [ ]:
import os

# # If you want to only use model files already downloaded, uncomment:
# os.environ["HF_HUB_OFFLINE"] = "1"

import torch
import transformers
from transformers import pipeline
# disable verbosity
from transformers import logging
logging.set_verbosity_error()


print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

We use `Qwen/Qwen2.5-0.5B-Instruct`, a smaller instruction-tuned model suitable for this local demonstration.
It is _very_ small but sufficient for demonstrating local inference with the Transformers pipeline.
<!-- It differs in size from the 72B model in the API notebook, so do not interpret differences in answers as effects of the client alone. -->

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

::: {.callout-warning title="Offline mode requires cached files"}

The model weights, tokenizer, and configuration files must already be cached.
If you need to download them, remove the offline-setting line and restart the kernel before running this notebook.
Alternatively, set `MODEL_ID` to a local folder containing the downloaded model.
No inference API token or API credit is needed for this public model.

:::

### Choose a compute device

A GPU can speed up generation.
CUDA supports compatible NVIDIA GPUs; MPS supports compatible Apple Silicon systems.
The CPU is the fallback and may be slower.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

### Create the pipeline

Creating a pipeline loads the model into memory.
Unlike constructing an API client, this can take time and several gigabytes of memory.
We create it once and reuse it throughout the notebook.

In [ ]:
chat = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device=device,
    dtype="auto",
)

::: {.callout-tip title="Device or memory problems?"}

If the accelerator does not support the checkpoint's precision, try `dtype=torch.float32`.
If accelerator memory is insufficient and you get a `RuntimeError` or out-of-memory error, try `device="cpu"`; the model still requires enough system memory.

:::

## First completion: the text-generation pipeline

### Define the conversation

As in the API notebook, a chat prompt is a list of messages with `role` and `content` keys:

- A `system` message gives the task instructions.
- A `user` message supplies the text to analyse.
- An `assistant` message represents a previous model response.

In [ ]:
post_text = (
    "Great news for our town! Today the council approved funding for a new "
    "public library. I am delighted that we can give everyone more places "
    "to learn, meet, and connect. Proud of what we have achieved together!"
)
print(post_text)

::: {.callout-tip title="Fictitious post"}

This post is invented for teaching and is not attributed to a real politician.
We classify the sentiment expressed by its author, not our own opinion of the politician or policy.

:::

The task instruction asks for a label without an explanation.
At this point, that is a prompt instruction, not a restriction on which tokens the model can generate.

In [ ]:
task_instruction = """\
You classify the sentiment expressed in politicians' social media posts. 

Identify the sentiment as positive, negative, or neutral. 

Only return the classification result without any explanation.\
"""

messages = [
    {"role": "system", "content": task_instruction},
    {"role": "user", "content": post_text},
]

::: {.callout-tip title="Using system and user messages"}

Separating instructions from the text makes the prompt easier to inspect and reuse.
Passing this list to the pipeline activates chat formatting automatically.
We do not need to manually insert role delimiters or special tokens.

:::

### Generating a response

Our first call uses the pipeline as-is.

In [ ]:
result = chat(messages)

### Inspect the response step by step

The pipeline returns a list of dictionaries. 
For this single conversation and one returned completion, the list has one element.

In [ ]:
print("Type:", type(result))
print("Number of completions:", len(result))
print("First completion type:", type(result[0]))
print("Keys:", list(result[0].keys()))

By default, `generated_text` contains the conversation including the newly generated assistant message.

In [ ]:
print(*result[0]["generated_text"], sep="\n")

The last message is the **assistant's answer** with the classification.

The index `-1` selects the last element of a Python list.

In [ ]:
assistant_message = result[0]["generated_text"][-1]
print(assistant_message)
print(assistant_message["content"])

::: {.callout-tip title="Congratulations"}

You have used a locally running LLM to complete a text annotation task!
The returned text is the model's classification choice, expressed as generated text.

:::


## Inspecting token probabilities

Generative LLMs produce text by predicting the next token in a sequence based on the input and previously generated tokens.
This next-token prediction is **probabilistic**, meaning the model assigns a probability to each possible next token based on the current context.

::: {.callout-note title="Next token probability example"}

Say the model has so far generated the sequence of tokens:

`["The", "cat", "sat", "on", "the"]`

The model then computes what's the probability of its token to be next in this sequence.

This will likely give a token like "table" or "mat" a higher probability than less contextually appropriate tokens, like "banana" or "car".

:::

Specifically, at each generation step, a model computes scores called **logits** for possible next tokens.
These correspond to the probabilities of each token in the model's vocabulary of being returned at the given generation step.

Inspecting a model's generation logits and token probabilities helps us understand the model's confidence in its predictions.

### Request scores and raw logits

We can configure a call to the text generation pipeline instance `chat` to return the generated token probabilites by specifying to arguments:

- `return_dict_in_generate=True` ensures the pipeline returns a dictionary containing scores and logits.
- `output_logits=True` requests the pipeline to include the raw model logits for each generated token.
<!-- - `output_scores=True` requests the pipeline to include the decoding scores for each generated token. -->

::: {.callout-note title="Logits vs. scores"}

**Logits** are unnormalized model scores, not probabilities.
They have not been normalized into probabilities.
Nor have they been affected by any sampling controls (discussed below).

(Also) pass `output_scores=True` to request decoding scores.

:::

**Logits** are the raw, unnormalized scores output by the model for each possible next token.
As in a logistic or multinomial regression, logits can be converted to probabilities.

In [ ]:
result = chat(messages, return_dict_in_generate=True, output_logits=True)
print(list(result[0].keys()))
print(result[0]["generated_text"][-1]["content"])

You can see that in addition to the *generated_text* containing the assistant's response message, the result also contains a key named `"logits"` holding the raw logits for each generated token.

### Inspect the logits

The logits element is a nested lists indexed by generation step and vocabulary token ID.

In [ ]:
print(type(result[0]["logits"]))
print(len(result[0]["logits"]))
print(len(result[0]["logits"][0]))

Here, we get a list with two elements, one per generated token.

The first element corresponds to the first generated token, and the second element corresponds to the end-of-sequence token.

For the first token, the corresponding logits are in the first element of the list.

There are as many elements in the list as there are tokens in the model's vocabulary (here 151936).

We convert the logits list of list to a PyTorch tensors for calculations:

In [ ]:
logits = torch.tensor(result[0]["logits"], dtype=torch.float32)

print("Logits shape:", logits.shape)

The first index selects a generation step, not a prompt token.
The second dimension contains the model's output vocabulary scores, including special tokens.

#### Which tokens are we looking for?

We have instructed the model to classify the post as either positive, negative, or neutral.

So we are most interested in the corresponding tokens' probabilities in the model's generated output.

Which are the corresponding tokens?

This we can figure out using the tokenizer to convert the sentiment labels into token IDs.

In [ ]:
sentiment_labels = ["positive", "negative", "neutral"]
sentiment_token_ids = [
    chat.tokenizer.encode(label, add_special_tokens=False)[0] 
    for label in sentiment_labels
]
print("Sentiment token IDs:", sentiment_token_ids)

We can index the `logits` using the sentiment token IDs to extract the relevant scores.
THe first generated token is the classification label the model produced.
So we index with 0 in the tensor's first dimension to get the logits for the first generated token.

In [ ]:
first_token_logits = logits[0]
sentiment_logits = first_token_logits[sentiment_token_ids]
print("Sentiment logits for the first generated token:", sentiment_logits)

These logits are not directly interpretable as probabilities; we need to apply the softmax function to convert them into probabilities.

### From logits to probabilities

For logits $z_i$, softmax gives a probability distribution over the vocabulary:

$$
P(t_i \mid \text{prompt}) = \frac{\exp(z_i)}{\sum_j \exp(z_j)}.
$$

The denominator includes **all vocabulary entries**, not just the three sentiment labels.
Using PyTorch's softmax avoids manually exponentiating large scores.

In [ ]:
first_token_logits = logits[0]  # First generation step, all vocabulary entries.
first_token_logprobs = torch.log_softmax(first_token_logits, dim=-1)
first_token_probs = torch.exp(first_token_logprobs)

print("Probability sum:", first_token_probs.sum().item())
# note: the sum is not necessarily 1 because of numerical precision issues.

So with the probabilities, we can look at the selection probabilities of the sentiment label categories:

In [ ]:
sentiment_probs = first_token_probs[sentiment_token_ids]
for label, prob in zip(sentiment_labels, sentiment_probs.tolist()):
    print(f"{label}: {prob:.6f}")

This reflects that the model responded with the sentiment label with the highest token probability.

Note that the three label class tokens' **probabilities do not sum to one.**
This happens because the model makes predictions for every token in its vocabulary, and the softmax normalization considers the entire vector of predicted probabilities.
So the remaining probability mass is on other tokens.

To see this, let's get the top-10 tokens with the highest probabilities:

In [ ]:
k = 10
# get index of top-k tokens
top_probs, top_ids = first_token_probs.topk(k)
top_tokens = chat.tokenizer.convert_ids_to_tokens(top_ids.tolist())
for token, prob in zip(top_tokens, top_probs.tolist()):
    print(f"{token}: {prob:.6f}")

### Visualize next-token prediction

This chart is analogous to inspecting BERT's candidate tokens in the Day 1 fill-mask exercise.
The bars are model probabilities, not observed frequencies from repeated generations.

In [ ]:
import matplotlib.pyplot as plt

labels = [repr(chat.tokenizer.decode([i])) for i in top_ids.tolist()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(k), top_probs.tolist())
ax.set_yticks(range(k), labels=labels)
ax.invert_yaxis()  # Put the most likely token at the top.
ax.set_xlabel("Next-token probability (full-vocabulary normalization)")
ax.set_title("What token comes next after our sentiment prompt?")
fig.tight_layout()
plt.show()

::: {.callout-note title="Connection to BERT's fill-mask predictions"}

In Day 1, BERT assigned scores to vocabulary tokens at a **masked position**, using context on both sides.
Here, Qwen assigns scores to vocabulary tokens at the **next position**, using the preceding prompt.

The shared idea is a distribution over candidate tokens.
The context and training objective differ: masked-token prediction versus causal next-token prediction.
To generate a longer answer, Qwen appends the chosen token and repeats the process.

:::

### Optional: normalize over the three selected labels

For a comparison restricted to those three token IDs, we can apply softmax only to their logits.
This produces a **conditional distribution given that the next token is one of those three tokens**.

In [ ]:
conditional_label_probs = torch.softmax(first_token_logits[sentiment_token_ids], dim=-1)

for label, probability in zip(sentiment_labels, conditional_label_probs.tolist()):
    print(f"{label}: {probability:.6f}")

print("Conditional sum:", conditional_label_probs.sum().item())

::: {.callout-warning title="Conditional probability is not classification accuracy"}

Normalizing over three labels discards probability mass assigned to all other tokens.
It does not constrain generation, and it is not an estimate of the probability that the classification is correct.
A high value can occur even when all three exact tokens had low original probability.
Evaluate classification quality against labelled examples.

:::

## Controlling text generation

Generative LLMs produce text by predicting the next token in a sequence based on the input and previously generated tokens.
This iterative, auto-regressive process is controlled by so-called **generation parameters**, related to _sampling_ (discussed below) and the length of the generated sequence (_max_new_tokens_ and _max_length_).

### Controlling the sequence length

We can control the length of the generated sequence using the `max_new_tokens` and `max_length` parameters:

- `max_length` specifies the maximum total length of the sequence, including the prompt.
- `max_new_tokens` specifies the maximum number of new tokens to generate.

The `max_new_tokens` parameter is helpful if we know that the available answer categories in a classification or multiple choice task are limited to a maximum number of tokens.
For example, the tokens' "positive", "negative", and "neutral" are all single-token answers in a classification task, which we have already confirm above:

In [ ]:
sentiment_token_ids

So we can limit the model's output to a single token, which is sufficient for this classification task.

In [ ]:
result = chat(messages, max_new_tokens=1)
result[0]["generated_text"][-1]["content"]

::: {.callout-warning title="One token is not necessarily one label"}

A token can be a word fragment, whitespace, punctuation, or a special token.
Even if a label can be encoded as one token, the model might first generate a newline or a different capitalization.
`max_new_tokens=1` is useful for this demonstration; it is not a general guarantee of a complete classification.

:::

### Controlling decoding

Remember that LLMs like Qwen **generate text** by predicting one token at a time based on the preceding context.
Choosing which token to return at each step of the generation process is referred to as **decoding**.

#### Greedy decoding

At each step, the model computes a score for every possible next token.
With **greedy decoding**, it selects the token with the highest decoding score.
In Transformers, `do_sample=False` together with `num_beams=1` selects greedy decoding.

We now set `max_new_tokens=1` to inspect just the first generation step.
The prompt length does not count toward this limit.

In [ ]:
result = chat(messages, do_sample=False, max_new_tokens=1)
print(result[0]["generated_text"][-1]["content"])

The following experiment shows that turning of sampling (i.e., setting `do_sample=False`) enables greedy decoding, leading to consistent outputs.

For this experiment, we define a random number generation prompt that let's the model choose a number between 1 and 100:

In [ ]:
random_number_choice_prompt = (
    "Choose an integer between 1 and 100. "
    "Pick one you like best. "
    "Only return the number and nothing else."
)

Running the model without switching off sampling, you likely get different responses across the five repeated requests:

In [ ]:
for _ in range(5):
    result = chat(
        [{"role": "user", "content": random_number_choice_prompt}],
        do_sample=True, # default
        max_new_tokens=1
    )
    print(result[0]["generated_text"][-1]["content"])

But if you enable greedy decoding by setting `do_sample=False`, the model will consistently return the same response across repeated requests:

In [ ]:
for _ in range(5):
    result = chat(
        [{"role": "user", "content": random_number_choice_prompt}],
        do_sample=False, # enable greedy decoding
        max_new_tokens=1
    )
    print(result[0]["generated_text"][-1]["content"])

::: {.callout-note title="Maybe use a generation configuration instead"}

The pipeline supplies a `generation_config` internally.
Recent Transformers versions warn when generation overrides are passed alongside that object.
Keeping the settings together avoids that warning and makes our choices easy to inspect.

```python
from copy import deepcopy

generation_config = deepcopy(chat.generation_config)

# then only modify the parameters you want to change, e.g.
generation_config.max_new_tokens = 1
generation_config.do_sample = False
```

#### Sampling

When not decoding greedily, the model samples from the probability distribution over the next token instead of always picking the most likely one.
Sampling settings affect which token is selected from those possibilities.

Details are described [here](https://huggingface.co/docs/transformers/en/generation_strategies) and we'll discuss only some important ones.


##### Temperature: how concentrated is the probability distribution on the most-likely token?

Remember that the model generates a probability distribution over all tokens in its vocabulary to determine the next tokens at each step.

The **temperature** setting modulates how peak or flat this probability distribution is:

- at a value of 1, the probability distribution is used as-is.
- values below 1 make the probability distribution sharper, favoring higher-probability tokens more strongly.
- values above 1 make the probability distribution flatter, increasing the likelihood of sampling lower-probability tokens.

Typical values are between 0.7 and 1.2 for most natural language generation tasks.
For scientific use such as text classification, researchers often use lower values, including 0.0.

Specifically, for a non-zero temperature $T$, the usual temperature transformation is:

$$
p_i = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)},
$$

where $z_i$ is the logit for token $i$ and $p_i$ is the resulting probability for token $i$.

The following graphic simulates how different temperature settings affect the probability distribution over six tokens, "1" through "6".

In [ ]:
import numpy as np

# sample default probability distribution for six tokens
probs = np.array([0.4, 0.2, 0.15, 0.1, 0.1, 0.05])

import matplotlib.pyplot as plt

def plot_temperature_effect(probs, temperatures):
    tokens = np.arange(1, len(probs) + 1)
    fig, axes = plt.subplots(
        1, len(temperatures), figsize=(3 * len(temperatures), 4), sharey=True
    )
    axes = np.atleast_1d(axes)
    for ax, T in zip(axes, temperatures):
        adjusted_probs = np.exp(np.log(probs) / T)
        adjusted_probs /= adjusted_probs.sum()
        color = 'black' if T == 1.0 else 'tab:blue'
        ax.bar(tokens, adjusted_probs, color=color)
        ax.set_title(f'T={T}')
        ax.set_xlabel('Token')
        ax.set_xticks(tokens)
    axes[0].set_ylabel('Probability')
    fig.suptitle('Effect of Temperature on Token Probabilities')
    fig.tight_layout()
    plt.show()

plot_temperature_effect(probs, [0.1, 0.5, 1.0, 1.5, 2.0])

::: {.callout-warning title="temperature=0.0!?"}

Many researchers write in their reports and recommend to set the temperature to 0.0 to effectively make the model deterministic, always choosing the highest-probability token at each step.

Given the formula above, setting T to 0.0 is not possible though.

What happens instead is that if the user specified `temperature=0` in their API request, sampling is _disabled_ (i.e., greedy decoding _enabled_).

This achieves the same as setting `do_sample=False` or `top_k=1` (see below).

:::

##### Top-k: how many top candidates are considered?

Again, remember that the model generates a probability distribution over all tokens in its vocabulary to determine the next tokens at each step.

But when setting `top_k=k`, only the top `k` tokens are considered for sampling, and all others are ignored.

Typicall values are 5, 10, 20, 50, or 100.

Setting `top_k` helps to limit the model's choices to a manageable set of plausible options while still allowing some randomness.

##### Top-p: how many alternatives remain eligible?

Nucleus sampling (`top_p`) retains the smallest set of highest-probability tokens whose cumulative probability reaches the threshold. 
If probabilities are 0.60, 0.25, 0.10, and 0.05, a threshold of 0.80 retains the first two candidates, whose combined probability is 0.85. Sampling then operates on the retained set.

## Generation settings at a glance

| Setting | Purpose in the local pipeline |
|---|---|
| `max_new_tokens=1` | Generate at most one new token after the prompt. |
| `do_sample=False` | Disable random sampling. With one beam, use greedy decoding. |
| `num_beams=1` | Use a single decoding path rather than beam search. |
| `temperature` | With sampling enabled, change how concentrated the distribution is. Use a positive value; use `do_sample=False` for greedy decoding, not `temperature=0`. |
| `top_k` | With sampling enabled, retain the highest-scoring candidate tokens; `0` disables this cutoff. |
| `top_p` | With sampling enabled, retain candidates up to a cumulative probability threshold; `1.0` disables this cutoff. |
| `repetition_penalty` | Adjust scores of previously occurring tokens; `1.0` is neutral. |
| `return_dict_in_generate=True` | Request richer internal generation output, allowing scores/logits to be retained. |
| `output_scores=True` | Return decoding scores for each generation step. |
| `output_logits=True` | Return raw model logits before generation-time score processing. |

These settings are documented in the [generation reference](https://huggingface.co/docs/transformers/main_classes/text_generation).

::: {.callout-note title="Getting inconsistent responses?"}

Greedy decoding removes random sampling; it does not guarantee identical results across hardware, numerical precision, or library versions.
It also does not guarantee a correct classification.
Record the model, prompt, configuration, and environment when comparing results.

:::

### Which settings are the defaults?

Defaults come from the model and pipeline.
Inspect the effective configuration instead of assuming sampling is disabled.

In [ ]:
for name in ("max_new_tokens", "max_length", "do_sample", "num_beams",
             "temperature", "top_k", "top_p", "repetition_penalty"):
    print(f"{name}: {getattr(chat.generation_config, name)}")

::: {.callout-warning title="Omitted ≠ disabled"}

An omitted setting falls back to a default.
In particular, the first call may use sampling and may produce more than one token.
A prompt asking for a short answer is not a token limit.

:::


## Practice and recap

1. Rewrite the fictional post to express disappointment about a delayed library opening. Rebuild `messages`, rerun the one-token call and the score-returning call, and inspect the top tokens.
2. Compare tokenization of `positive`, `Positive`, and ` positive`. Explain why their probabilities should not automatically be treated as the same event.
3. Set `max_new_tokens=3` in a copy of `score_config`. Inspect the three score rows if three tokens are generated. Why does each row condition on a different context?
4. Explain how BERT's fill-mask prediction and this pipeline's next-token prediction are similar, and how they differ.

We progressed from **chat completion with default settings**, to **inspection of the next-token distribution**, to **one-token greedy generation**.
The sentiment instruction changes the context in which the model predicts tokens; the generation mechanism remains next-token prediction.